# Data Processing

Notas do curso **Machine Learning Process** — **pré-processamento** de dados antes da modelagem.

> Complementa: [Data Collection.md](./Data%20Collection.md) · [Modeling Process.md](./Modeling%20Process.md)


In [14]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer

## Princípio central

> **Se os dados não são confiáveis, não confie no resultado do modelo.**

Pré-processamento não é burocracia — é garantir que o algoritmo receba informação **consistente**, **completa o suficiente** e **alinhada às suposições** do método escolhido.


## Perguntas antes de modelar

Responda para **cada algoritmo** candidato (não assuma regras genéricas):

| Pergunta | Por que importa |
|----------|-----------------|
| **O modelo lida com valores nulos?** | Regressão linear clássica **não** aceita null; muitas árvores **sim** (com ressalvas) |
| **Como outliers afetam a performance?** | Regressão linear é **sensível**; árvores costumam ser mais robustas |
| **O modelo exige scaling (normalização)?** | Regressão linear, SVM, k-NN, redes — em geral **sim**; árvores em geral **não** |
| **Quais suposições o modelo faz sobre os dados?** | Linearidade, independência, distribuição, homocedasticidade… |

Documente as respostas no plano de projeto — o pré-processamento **muda** conforme o modelo vencedor.


In [ ]:
# Código: nulls — dropping nulls
drop_df = df.copy().dropna()

X_d = drop_df[["age", "days_on_platform", "income"]]
y_d = drop_df["lifetime_value"]

n_train_d = int(len(X_d) * 0.8)
X_train_d = X_d.iloc[:n_train_d].copy()
y_train_d = y_d.iloc[:n_train_d].copy()
X_test_d = X_d.iloc[n_train_d:].copy()
y_test_d = y_d.iloc[n_train_d:].copy()

X_train_d.shape, X_test_d.shape


## Valores nulos (*missing values*)

### Comportamento por família de modelo

| Tipo de modelo | Valores nulos |
|----------------|---------------|
| **Regressão linear** (e muitos modelos lineares clássicos) | **Não podem** permanecer null — é preciso imputar, remover linhas ou usar pipeline que trate missing |
| **Modelos baseados em árvore** (Random Forest, XGBoost, LightGBM…) | **Podem lidar** com nulls de formas diferentes (imputação interna, branch “missing”, ou tratamento nativo dependendo da lib) |

> “Árvore aceita null” ≠ “pode ignorar qualidade dos dados”. Ainda vale auditar **por que** o dado está ausente.

### Estratégias comuns

| Estratégia | Quando usar |
|------------|-------------|
| **Remover** linhas/colunas | Poucos missing, MCAR, volume sobra |
| **Imputar** (média, mediana, moda) | Numérico simples; cuidado com vazamento no treino |
| **Flag** `was_missing` | Ausência pode ser sinal (ex.: renda não informada) |
| **Imputação avançada** (KNN, iterativa) | Relações entre features; mais custo |

**Regra:** impute no **treino** e aplique a mesma regra no **teste/produção** (fit apenas no train).


In [ ]:
# Código: outliers



## Outliers (pontos desviantes)

**Outliers** = pontos muito distantes do restante da distribuição.

### Impacto na performance

| Modelo | Sensibilidade |
|--------|---------------|
| **Regressão linear** | **Alta** — poucos pontos extremos puxam a reta e inflam erro |
| **Árvores** | **Menor** — splits isolam extremos, mas ainda podem dominar folhas raras |
| **Distância / clustering** | Muito sensível — outliers viram clusters falsos |

### O que fazer

1. **Investigar** — erro de medição, fraude, whale user, evento real?  
2. **Não remover automaticamente** — remover tudo “estranho” pode apagar o sinal  
3. **Tratar quando necessário**  
   - cap/winsorize (teto no valor máximo)  
   - transformação log  
   - remover só após regra de negócio documentada  
   - modelos robustos (Huber, árvores, ensembles)

> Outliers às vezes **são** o negócio (top 1% de compradores). Remover sem entender distorce KPI e modelo.


In [ ]:
# Código: scaling



## Escalonamento (*scaling*)

Alguns algoritmos comparam features em **escalas diferentes** (idade 0–100 vs. renda 0–1.000.000).

| Precisa de scaling? | Exemplos |
|---------------------|----------|
| **Em geral sim** | Regressão linear (com regularização), logistic regression, SVM, k-NN, neural nets |
| **Em geral não** | Árvores de decisão, Random Forest, gradient boosting em árvores |

Técnicas: **StandardScaler** (média 0, desvio 1), **MinMaxScaler** (0–1), **RobustScaler** (usa mediana — útil com outliers).

Ajuste o scaler **só no conjunto de treino**, depois `transform` em validação e produção.


In [ ]:
# Código: pipeline / exercícios



## Suposições do modelo sobre os dados

Cada algoritmo traz **pressupostos** — o pré-processamento existe para aproximar os dados dessas expectativas (ou escolher outro algoritmo).

| Modelo | Suposições / implicações típicas |
|--------|----------------------------------|
| **Regressão linear** | Relação linear, erros independentes, homocedasticidade; **sem null**; sensível a outliers e escala |
| **Árvores** | Poucas suposições distributivas; lidam com não linearidade; **null** muitas vezes OK; escala irrelevante |
| **Logistic / classificação linear** | Como linear + alvo binário/multinomial; scaling recomendado |

Se os dados violam suposições de forma grave, prefira outro modelo ou transforme features (log, bins, interações).


## Fluxo sugerido de pré-processamento

```
  Auditar qualidade → Tratar nulls (conforme modelo) → Outliers (investigar → decidir)
       → Encoding categórico → Scaling (se necessário) → Validar no hold-out
```

| Etapa | Pergunta-chave |
|-------|----------------|
| Qualidade | Os dados são **confiáveis**? Definições corretas? |
| Nulls | O modelo escolhido **aguenta** null? Se não, qual imputação? |
| Outliers | São erro ou sinal? Como afetam a métrica de negócio? |
| Scaling | Features na mesma escala são obrigatórias? |
| Suposições | Linear? temporal? precisa de balanceamento de classes? |


## Checklist — data processing

- [ ] Qualidade dos dados validada (confiança na fonte)  
- [ ] Resposta: **modelo lida com null?** — estratégia definida  
- [ ] Outliers **investigados** — impacto na performance avaliado  
- [ ] Decisão documentada: manter, cap, transformar ou remover  
- [ ] **Scaling** aplicado só se o algoritmo exigir  
- [ ] **Suposições** do modelo listadas e checadas (EDA / testes)  
- [ ] Pipeline reproduzível: mesmo tratamento em treino e produção


## Suas notas

- 
-


/Users/iancalixto/Documents/GitHub/ML_Process_Course/5_data_preprocessing/5_1_missing_values/clv_data.csv
(5000, 8)


,Unnamed: 0,id,age,gender,income,days_on_platform,city,purchases
0,0,0,NaN,Male,126895,14.0,San Francisco,0
1,1,1,NaN,Male,161474,14.0,Tokyo,0
2,2,2,24.0,Male,104723,34.0,London,1
3,3,3,29.0,Male,43791,28.0,London,2
4,4,4,18.0,Female,132181,26.0,London,2


In [18]:
import random

df = pd.read_csv("../drive/My Drive/ML_Process_Data_Files/Section_5_Data_Preprocessing/clv_data.csv")

df['lifetime_value'] = df['purchases'] * 20

df.head()

,Unnamed: 0,id,age,gender,income,days_on_platform,city,purchases,lifetime_value
0,0,0,NaN,Male,126895,14.0,San Francisco,0,0
1,1,1,NaN,Male,161474,14.0,Tokyo,0,0
2,2,2,24.0,Male,104723,34.0,London,1,20
3,3,3,29.0,Male,43791,28.0,London,2,40
4,4,4,18.0,Female,132181,26.0,London,2,40


In [19]:
random.randint(5, 50)

21

In [20]:
df.isnull().sum()

Unnamed: 0             0
id                     0
age                 2446
gender                 0
income                 0
days_on_platform     141
city                   0
purchases              0
lifetime_value         0
dtype: int64

### Checking Null Values

In [21]:
def nulls_summary_table(df):
    """
    Summarize null values in a DataFrame.
    """
    null_values = pd.DataFrame(df.isnull().sum())
    null_values[1] = null_values[0] / len(df)
    null_values.columns = ['null_count', 'null_pct']
    return null_values

nulls_summary_table(df)

,null_count,null_pct
Unnamed: 0,0,0.0000
id,0,0.0000
age,2446,0.4892
gender,0,0.0000
income,0,0.0000
days_on_platform,141,0.0282
city,0,0.0000
purchases,0,0.0000
lifetime_value,0,0.0000


In [22]:
drop_df = df.copy()
drop_df = drop_df.dropna()

In [25]:
# Split já feito na célula "Código: nulls" — esta célula ficou vazia de propósito.


((1476, 3), (1476,))

### Mean / Median / Mode imputation

Calcule **mean**, **median** e **mode** no **treino** e use esses valores para `fillna` no treino e no teste (nunca use estatísticas do teste para imputar).

In [26]:
m_df = df.copy()

X_m = m_df[["age", "days_on_platform", "income"]].copy()
y_m = m_df["lifetime_value"].copy()

n_train = 4000
X_train_m = X_m.iloc[:n_train].copy()
y_train_m = y_m.iloc[:n_train].copy()
X_test_m = X_m.iloc[n_train:].copy()
y_test_m = y_m.iloc[n_train:].copy()

X_train_m.shape, X_test_m.shape


((4000, 3), (4000,))

In [29]:
cols_impute = ["age", "days_on_platform"]

# Média, mediana e moda (só no treino)
impute_stats = pd.DataFrame(
    {
        col: {
            "mean": X_train_m[col].mean(),
            "median": X_train_m[col].median(),
            "mode": stats.mode(X_train_m[col].dropna()).mode[0],
        }
        for col in cols_impute
    }
).T

impute_stats

# Mean imputation
X_train_mean = X_train_m.copy()
X_test_mean = X_test_m.copy()
for col in cols_impute:
    fill = impute_stats.loc[col, "mean"]
    X_train_mean[col] = X_train_mean[col].fillna(fill)
    X_test_mean[col] = X_test_mean[col].fillna(fill)

# Median imputation
X_train_median = X_train_m.copy()
X_test_median = X_test_m.copy()
for col in cols_impute:
    fill = impute_stats.loc[col, "median"]
    X_train_median[col] = X_train_median[col].fillna(fill)
    X_test_median[col] = X_test_median[col].fillna(fill)

# Mode imputation
X_train_mode = X_train_m.copy()
X_test_mode = X_test_m.copy()
for col in cols_impute:
    fill = impute_stats.loc[col, "mode"]
    X_train_mode[col] = X_train_mode[col].fillna(fill)
    X_test_mode[col] = X_test_mode[col].fillna(fill)

X_train_mean.isnull().sum()


age                 0
days_on_platform    0
income              0
dtype: int64

In [30]:
# (imputação mean/median/mode na célula anterior — use X_train_mean / X_test_mean)

/Users/iancalixto/Documents/GitHub/ML_Process_Course/.venv/lib/python3.10/site-packages/pandas/core/indexing.py:1773: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(ilocs[0], value, pi)


In [31]:
# Imputação multivariada (SimpleImputer — estável neste ambiente)
# Nota: IterativeImputer/KNNImputer podem derrubar o kernel com sklearn 1.0.2 no macOS.
from sklearn.impute import SimpleImputer

r_df = df.copy()
feature_cols = ["age", "days_on_platform", "income"]

X_r = r_df[feature_cols].copy()
y_r = r_df["lifetime_value"].copy()

X_train_r = X_r.iloc[:n_train].copy()
y_train_r = y_r.iloc[:n_train].copy()
X_test_r = X_r.iloc[n_train:].copy()
y_test_r = y_r.iloc[n_train:].copy()

imp_r = SimpleImputer(strategy="mean")
X_train_r = pd.DataFrame(
    imp_r.fit_transform(X_train_r),
    columns=feature_cols,
    index=X_train_r.index,
)
X_test_r = pd.DataFrame(
    imp_r.transform(X_test_r),
    columns=feature_cols,
    index=X_test_r.index,
)

X_train_r.isnull().sum()


: 